# DBNL Rao pipeline — Colab runner

Code comes from GitHub (`trister95/unseen-semantic-diversity`).
Data + secrets come from Drive (`MyDrive/dbnl/`).

Layout on the VM:
- `/content/code/` — clone of the repo (`functional_diversity/rao/...` inside)
- `/content/data/dbnl_txt_files/` — extracted from Drive tar.gz
- `/content/drive/MyDrive/dbnl/` — persistent Drive store (metadata CSV, .env, work backup)
- `/content/work/` — outputs; explicitly copied to Drive after each stage

## 1. Mount Drive, load `.env`, check GPU

In [3]:
import os
from google.colab import drive
drive.mount('/content/drive')

ENV_PATH = '/content/drive/MyDrive/dbnl/.env'
with open(ENV_PATH) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        k, _, v = line.partition('=')
        os.environ[k.strip()] = v.strip().strip('"').strip("'")
assert 'HF_TOKEN' in os.environ, '.env loaded but no HF_TOKEN found'
print('HF_TOKEN starts with:', os.environ['HF_TOKEN'][:8] + '…')

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

Mounted at /content/drive
HF_TOKEN starts with: hf_uJgVW…
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


## 2. Clone (or pull) the repo

Public repo: HTTPS clone, no token needed.
Re-running pulls instead of cloning, so cell is idempotent.

In [4]:
REPO_URL = 'https://github.com/trister95/unseen-semantic-diversity.git'
REPO_DIR = '/content/code'

import os
if os.path.isdir(REPO_DIR + '/.git'):
    !cd $REPO_DIR && git pull --ff-only
else:
    !git clone $REPO_URL $REPO_DIR

%cd $REPO_DIR
!ls rao

Cloning into '/content/code'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 58 (delta 19), reused 45 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 182.03 KiB | 6.28 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/code
build_corpus.py    occurrence_rao.py  README.md
embed_mentions.py  rao_q.py	      START_HERE.md


## 3. Install dependencies

In [5]:
!pip install -q -U transformers spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 60.7 MB/s eta 0:00:0000:010:01


## 4. Extract data to local VM disk

Reading from `/content/drive/` during the pipeline is much slower than `/content/`. One-time tar extract per session.

In [6]:
if not os.path.exists('/content/data/dbnl_txt_files'):
    !mkdir -p /content/data
    !tar -xzf /content/drive/MyDrive/dbnl/dbnl_txt_files.tar.gz -C /content/data
print('files:', len(os.listdir('/content/data/dbnl_txt_files')))

files: 7212


## 5. Restore prior work from Drive (resume) + start checkpoint loop

If a previous run was interrupted, copying the DB from Drive lets `build_corpus.py` resume from where it stopped. The background loop pushes the DB back to Drive every 5 min so we don't lose work to a session timeout.

In [ ]:
!mkdir -p /content/work
!cp -n /content/drive/MyDrive/dbnl/work/dbnl.sqlite /content/work/ 2>/dev/null || echo '(no prior DB — fresh run)'
!cp -n /content/drive/MyDrive/dbnl/work/dbnl_empty.sqlite /content/work/ 2>/dev/null || true

# Checkpoint loop: snapshot the live SQLite DBs to Drive every 2 min using
# SQLite's online-backup API (pure Python — no sqlite3 CLI binary needed, which
# isn't always installed). The DBs are opened WAL-mode by build_corpus.py;
# .backup() yields a consistent single-file snapshot from a live DB, unlike a
# plain cp (which can miss the -wal file or copy a torn file mid-checkpoint).
import os, time, threading, sqlite3

DRIVE_WORK = '/content/drive/MyDrive/dbnl/work'
PAIRS = [
    ('/content/work/dbnl.sqlite',       f'{DRIVE_WORK}/dbnl.sqlite'),
    ('/content/work/dbnl_empty.sqlite', f'{DRIVE_WORK}/dbnl_empty.sqlite'),
]

def _snapshot(src, dst):
    if not os.path.exists(src):
        return
    try:
        s = sqlite3.connect(src, timeout=30)
        s.execute('PRAGMA busy_timeout=30000;')
        d = sqlite3.connect(dst)
        with d:
            s.backup(d)          # online backup: safe on a live, being-written DB
        d.close(); s.close()
    except Exception as e:
        print('checkpoint error', src, '->', e, flush=True)

def _loop():
    os.makedirs(DRIVE_WORK, exist_ok=True)
    while True:
        for src, dst in PAIRS:
            _snapshot(src, dst)
        time.sleep(120)

_checkpoint = threading.Thread(target=_loop, daemon=True)
_checkpoint.start()
print('checkpoint thread started (every 120s, online-backup API)')

## 7. Stage 1 — real run

Year range editable. Estimated wall times on T4 @ ~0.29 files/s:
- 1600-1700: ~40 min (673 files)
- 1500-1700: ~50-120 min
- 1500-1800: ~5 h

Resumes via the `docs` table — re-launching after a session timeout picks up where it left off.

In [ ]:
!python rao/build_corpus.py \
    --input-dir /content/data/dbnl_txt_files \
    --db /content/work/dbnl.sqlite \
    --empty-db /content/work/dbnl_empty.sqlite \
    --metadata /content/drive/MyDrive/dbnl/dbnl_metadata.csv \
    --min-year 1500 --max-year 1700 \
    --log-every 50 --device cuda:0 \
    --ner-batch-size 512 \
    --ner-buffer-segments 1024

In [7]:
!cp /content/work/dbnl.sqlite /content/drive/MyDrive/dbnl/work/
!cp /content/work/dbnl_empty.sqlite /content/drive/MyDrive/dbnl/work/
!cp /content/work/dbnl.sqlite /content/work/dbnl_1500_1700.sqlite


In [ ]:
#run this later for 1700-1800 ;after thta 1800-1900 at some point
!python rao/build_corpus.py \
    --input-dir /content/data/dbnl_txt_files \
    --db /content/work/dbnl.sqlite \
    --empty-db /content/work/dbnl_empty.sqlite \
    --metadata /content/drive/MyDrive/dbnl/dbnl_metadata.csv \
    --min-year 1700 --max-year 1800 \
    --log-every 50 --device cuda:0 \
    --ner-batch-size 512 \
    --ner-buffer-segments 1024
    
# Explicit final sync (background loop catches it within 2 min, but be explicit before moving on)
!cp /content/work/dbnl.sqlite /content/drive/MyDrive/dbnl/work/dbnl_1500_1800.sqlite
!cp /content/work/dbnl_empty.sqlite /content/drive/MyDrive/dbnl/work/

## 8. Quick SQLite sanity peek

In [8]:
import sqlite3
con = sqlite3.connect('/content/work/dbnl.sqlite')
print('docs     :', con.execute('SELECT COUNT(*) FROM docs').fetchone()[0])
print('sentences:', con.execute('SELECT COUNT(*) FROM sentences').fetchone()[0])
print('mentions :', con.execute('SELECT COUNT(*) FROM mentions').fetchone()[0])

bad = con.execute("""
    SELECT COUNT(*) FROM mentions m JOIN sentences s ON m.sentence_id = s.sentence_id
    WHERE m.doc_id != s.doc_id
""").fetchone()[0]
print('orphan mentions (should be 0):', bad)

print('\n--- top 15 animal surface forms ---')
for txt, n in con.execute(
    'SELECT text, COUNT(*) c FROM mentions GROUP BY text ORDER BY c DESC LIMIT 15'
):
    print(f'  {n:6d}  {txt}')
con.close()

docs     : 766
sentences: 93794
mentions : 143179
orphan mentions (should be 0): 0

--- top 15 animal surface forms ---
    2816  beesten
    2258  dieren
    1960  hert
    1844  dier
    1639  schapen
    1631  Lam
    1538  vee
    1326  beest
    1182  Vee
    1115  Dieren
    1104  visschen
    1089  Visch
    1054  honden
    1006  Visschen
     938  Leeuw


## 9. Stage 2 — occurrence embeddings

In [9]:
!python rao/embed_mentions.py \
    --db /content/work/dbnl.sqlite \
    --output /content/work/occurrences.npz \
    --device cuda:0 \
    --batch-size 512

!cp /content/work/occurrences.npz /content/drive/MyDrive/dbnl/work/

import numpy as np
z = np.load('/content/work/occurrences.npz')
print({k: z[k].shape for k in z.files})

143179 mentions across 93794 sentences; device=cuda:0
config.json: 100% 337/337 [00:00<00:00, 1.72MB/s]
vocab.txt: 225kB [00:00, 9.52MB/s]
pytorch_model.bin: 100% 441M/441M [00:04<00:00, 89.6MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 14851.98it/s]
[transformers] BertModel LOAD REPORT from: emanjavacas/GysBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 


## 10. Stage 3 — occurrence-level Rao Q per decade

In [10]:
!python rao/occurrence_rao.py \
    --occurrences /content/work/occurrences.npz \
    --metadata /content/drive/MyDrive/dbnl/dbnl_metadata.csv \
    --output /content/work/rao_occurrence.jsonl \
    --plot-output /content/work/rao_occurrence.png

!cp /content/work/rao_occurrence.jsonl /content/drive/MyDrive/dbnl/work/
!cp /content/work/rao_occurrence.png /content/drive/MyDrive/dbnl/work/

print('\n--- JSONL ---')
!cat /content/work/rao_occurrence.jsonl

loaded 143179 occurrences (dim 768) from /content/work/occurrences.npz
metadata: 5936 ti_ids with parseable jaar
wrote 20 groups to /content/work/rao_occurrence.jsonl

per-group results:
         group    n_occ  n_types     Rao Q   cent norm
          1500      120       56    0.4224      0.7600
          1520      152      109    0.4937      0.7115
          1530      859      310    0.4619      0.7336
          1540     1288      648    0.5665      0.6584
          1550      818      309    0.4975      0.7089
          1560     4879     1151    0.5227      0.6909
          1570      883      402    0.5342      0.6825
          1580     1966      735    0.5494      0.6713
          1590     4322     1107    0.5403      0.6780
          1600     5319     1820    0.5754      0.6517
          1610     8845     2843    0.5655      0.6591
          1620    12283     3494    0.5728      0.6536
          1630     6453     1537    0.5273      0.6876
          1640     9522     2362    0.5472 